# 06 — Clustering and Dimensionality Reduction


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Feature Matrix Construction

Normalised features: compile time, SLOC, file count, header depth, deps, codegen ratio, etc.


In [ ]:
import igraph as ig
import leidenalg
import numpy as np
from networkx.algorithms.community import louvain_communities, modularity
from sklearn.preprocessing import StandardScaler

# ── Reproduce community detection (self-contained, same logic as NB04) ──────
direct_edges = edge_df[edge_df['is_direct']]

G_und = nx.Graph()
G_und.add_nodes_from(G.nodes())
G_und.add_edges_from(zip(direct_edges['source_target'], direct_edges['dest_target']))

louvain_comms = louvain_communities(G_und, resolution=1.0, seed=42)
louvain_mod = modularity(G_und, louvain_comms)
louvain_node_comm = {}
for i, comm in enumerate(louvain_comms):
    for n in comm:
        louvain_node_comm[n] = i

ig_g = ig.Graph.from_networkx(G_und)
_node_names = ig_g.vs['_nx_name']
leiden_part = leidenalg.find_partition(ig_g, leidenalg.ModularityVertexPartition, seed=42)
leiden_node_comm = {_node_names[i]: leiden_part.membership[i] for i in range(ig_g.vcount())}
leiden_comms = [
    {n for n, c in leiden_node_comm.items() if c == cid}
    for cid in range(max(leiden_part.membership) + 1)
]
leiden_mod = modularity(G_und, leiden_comms)

if leiden_mod >= louvain_mod:
    node_comm = leiden_node_comm
    communities = leiden_comms
    comm_algo = 'Leiden'
    comm_mod = leiden_mod
else:
    node_comm = louvain_node_comm
    communities = louvain_comms
    comm_algo = 'Louvain'
    comm_mod = louvain_mod

n_communities = len(communities)
print(f"Community detection: {comm_algo}, {n_communities} communities, modularity = {comm_mod:.4f}")

# ── Filter to targets with compilable sources ───────────────────────────────
feat_df = target_df[target_df['compile_time_sum_ms'] > 0].copy().reset_index(drop=True)

excluded = target_df[target_df['compile_time_sum_ms'] <= 0][['cmake_target', 'target_type']]
print(f"\nTargets in feature matrix: {len(feat_df)}")
print(f"Excluded ({len(excluded)} targets with zero compile time):")
for _, row in excluded.iterrows():
    print(f"  {row['cmake_target']} ({row['target_type']})")

# ── Build feature matrix ────────────────────────────────────────────────────
FEATURES = [
    'compile_time_sum_ms', 'code_lines_total', 'file_count',
    'header_depth_max', 'preprocessed_bytes_total', 'object_size_total_bytes',
    'direct_dependency_count', 'transitive_dependency_count', 'direct_dependant_count',
    'git_commit_count_total', 'link_time_ms',
    'codegen_ratio', 'codegen_compile_time_sum_ms',
    'expansion_ratio_mean', 'gcc_template_time_sum_ms',
]

X_raw = feat_df[FEATURES].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Flag zero-variance features (e.g. gcc_template_time_sum_ms may be all zero)
zero_var = [f for f, v in zip(FEATURES, scaler.var_) if v == 0]
if zero_var:
    print(f"\nZero-variance features (constant across all targets): {zero_var}")
    print("  These contribute no discriminating power but are retained for completeness.")

print(f"\nFeature matrix: {X_scaled.shape[0]} targets × {X_scaled.shape[1]} features")

# ── Assign communities to filtered targets ──────────────────────────────────
feat_df['community'] = feat_df['cmake_target'].map(node_comm).fillna(-1).astype(int)
display(feat_df[['cmake_target', 'target_type', 'community'] + FEATURES[:5]])

## Dimensionality Reduction

PCA, then t-SNE/UMAP for 2D visualisation. Colour by community.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.cm as cm
from matplotlib.patches import Patch

# ── Community colour map ────────────────────────────────────────────────────
cmap = cm.tab10
comm_colour_map = {cid: cmap(cid / 9) for cid in range(n_communities)}
comm_colours = [comm_colour_map.get(c, (0.5, 0.5, 0.5, 1.0)) for c in feat_df['community']]
comm_legend = [
    Patch(facecolor=comm_colour_map[cid], label=f'C{cid} ({len(communities[cid])} targets)')
    for cid in range(n_communities)
]

# ── PCA ─────────────────────────────────────────────────────────────────────
n_components_full = min(len(FEATURES), len(feat_df))
pca_full = PCA(n_components=n_components_full, random_state=42)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
elbow_idx = int(np.searchsorted(cumvar, 0.90))  # first component reaching 90%

pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

n_pca_feed = min(5, n_components_full)
pca_feed = PCA(n_components=n_pca_feed, random_state=42)
X_pca_feed = pca_feed.fit_transform(X_scaled)

# ── t-SNE (on PCA-reduced features) ────────────────────────────────────────
perplexity = min(30, len(feat_df) - 1)
X_tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, max_iter=1000).fit_transform(X_pca_feed)

# ── UMAP ────────────────────────────────────────────────────────────────────
n_neighbors_umap = min(15, len(feat_df) - 1)
X_umap = umap.UMAP(n_neighbors=n_neighbors_umap, min_dist=0.3, random_state=42).fit_transform(X_scaled)

# ── Plots ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Top-left: Scree plot
ax = axes[0, 0]
components = np.arange(1, n_components_full + 1)
ax.bar(components, pca_full.explained_variance_ratio_, color='steelblue', alpha=0.7, label='Individual')
ax.plot(components, cumvar, 'o-', color='darkorange', label='Cumulative')
ax.axvline(elbow_idx + 1, color='red', linestyle='--', alpha=0.6,
           label=f'90% variance (PC{elbow_idx + 1})')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Scree Plot')
ax.legend()

# Top-right: PCA biplot with loading arrows
ax = axes[0, 1]
ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=comm_colours, s=60, zorder=5)
for i, name in enumerate(feat_df['cmake_target']):
    ax.annotate(name, (X_pca2[i, 0], X_pca2[i, 1]),
                textcoords='offset points', xytext=(3, 3), fontsize=7, ha='left')

# Loading arrows for top-6 features by magnitude
loadings = pca2.components_.T  # (n_features, 2)
loading_mag = np.sqrt(loadings[:, 0] ** 2 + loadings[:, 1] ** 2)
top6_idx = np.argsort(loading_mag)[-6:]
scale = max(np.abs(X_pca2).max(), 1.0) / max(np.abs(loadings[top6_idx]).max(), 1e-9) * 0.7
for idx in top6_idx:
    ax.annotate(
        '', xy=(loadings[idx, 0] * scale, loadings[idx, 1] * scale), xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='grey', lw=1.2),
    )
    ax.text(loadings[idx, 0] * scale * 1.08, loadings[idx, 1] * scale * 1.08,
            FEATURES[idx].replace('_', '\n', 1), fontsize=6, color='grey', ha='center')

ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('PCA Biplot (coloured by community)')
ax.legend(handles=comm_legend, fontsize=7, loc='best')

# Bottom-left: t-SNE
ax = axes[1, 0]
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=comm_colours, s=60, zorder=5)
for i, name in enumerate(feat_df['cmake_target']):
    ax.annotate(name, (X_tsne[i, 0], X_tsne[i, 1]),
                textcoords='offset points', xytext=(3, 3), fontsize=7, ha='left')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title(f't-SNE (perplexity={perplexity}, coloured by community)')

# Bottom-right: UMAP
ax = axes[1, 1]
ax.scatter(X_umap[:, 0], X_umap[:, 1], c=comm_colours, s=60, zorder=5)
for i, name in enumerate(feat_df['cmake_target']):
    ax.annotate(name, (X_umap[i, 0], X_umap[i, 1]),
                textcoords='offset points', xytext=(3, 3), fontsize=7, ha='left')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title(f'UMAP (n_neighbors={n_neighbors_umap}, coloured by community)')

plt.tight_layout()
plt.show()

# ── Summary ─────────────────────────────────────────────────────────────────
print("PCA explained variance:")
for i in range(min(5, n_components_full)):
    print(f"  PC{i+1}: {pca_full.explained_variance_ratio_[i]:.1%}  (cumulative: {cumvar[i]:.1%})")
print(f"\nt-SNE perplexity: {perplexity} (clamped to N-1 = {len(feat_df)-1})")
print(f"UMAP n_neighbors: {n_neighbors_umap}")

## Clustering

DBSCAN and hierarchical clustering. Compare with communities.


In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram

# ── k-distance plot for DBSCAN eps selection ────────────────────────────────
k = min(4, len(feat_df) - 1)
nn = NearestNeighbors(n_neighbors=k).fit(X_scaled)
k_distances = np.sort(nn.kneighbors(X_scaled)[0][:, -1])[::-1]

# Heuristic: find steepest gradient change for eps suggestion
gradients = np.diff(k_distances)
if len(gradients) > 1:
    grad_change = np.abs(np.diff(gradients))
    elbow_rank = np.argmax(grad_change) + 1
    eps_suggested = float(k_distances[elbow_rank])
else:
    eps_suggested = float(np.median(k_distances))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(k_distances) + 1), k_distances, 'o-', color='steelblue', markersize=4)
ax.axhline(eps_suggested, color='red', linestyle='--', alpha=0.6,
           label=f'Suggested eps = {eps_suggested:.2f}')
ax.set_xlabel('Points (sorted by distance)')
ax.set_ylabel(f'{k}-th Nearest Neighbour Distance')
ax.set_title(f'k-Distance Graph (k={k}) for DBSCAN eps Selection')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Suggested eps from elbow heuristic: {eps_suggested:.2f}")

# ── DBSCAN ──────────────────────────────────────────────────────────────────
dbscan = DBSCAN(eps=eps_suggested, min_samples=2).fit(X_scaled)
dbscan_labels = dbscan.labels_
n_dbscan_clusters = len(set(dbscan_labels) - {-1})
n_noise = (dbscan_labels == -1).sum()

print(f"\nDBSCAN (eps={eps_suggested:.2f}, min_samples=2): "
      f"{n_dbscan_clusters} clusters, {n_noise} noise points")

# ── Hierarchical clustering (Ward, matching community count) ────────────────
n_hclust = max(2, n_communities)
hclust = AgglomerativeClustering(n_clusters=n_hclust, linkage='ward')
hclust_labels = hclust.fit_predict(X_scaled)

print(f"Hierarchical Ward (n_clusters={n_hclust}): {n_hclust} clusters")

# ── Store labels ────────────────────────────────────────────────────────────
feat_df['dbscan_label'] = dbscan_labels
feat_df['hcluster'] = hclust_labels

# ── Comparison metrics vs communities ───────────────────────────────────────
comm_labels = feat_df['community'].values

# Filter to non-noise for DBSCAN ARI/AMI (noise targets have no cluster assignment)
non_noise_mask = dbscan_labels != -1
if non_noise_mask.sum() >= 2:
    ari_dbscan = adjusted_rand_score(comm_labels[non_noise_mask], dbscan_labels[non_noise_mask])
    ami_dbscan = adjusted_mutual_info_score(comm_labels[non_noise_mask], dbscan_labels[non_noise_mask])
else:
    ari_dbscan = ami_dbscan = float('nan')

ari_hclust = adjusted_rand_score(comm_labels, hclust_labels)
ami_hclust = adjusted_mutual_info_score(comm_labels, hclust_labels)

comparison = pd.DataFrame([
    {'method': f'DBSCAN (eps={eps_suggested:.2f})', 'n_clusters': n_dbscan_clusters,
     'n_noise': n_noise, 'ARI_vs_community': round(ari_dbscan, 4),
     'AMI_vs_community': round(ami_dbscan, 4)},
    {'method': f'Hierarchical Ward (n={n_hclust})', 'n_clusters': n_hclust,
     'n_noise': 0, 'ARI_vs_community': round(ari_hclust, 4),
     'AMI_vs_community': round(ami_hclust, 4)},
])
display(comparison)

print("\nInterpretation: ARI near 0 means feature-based clusters differ substantially from")
print("graph-topology communities — targets adjacent in the dependency graph do not")
print("necessarily share similar build cost profiles.")

# ── Three-way PCA biplot comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# DBSCAN
ax = axes[0]
cluster_cmap = cm.Set2
for label in sorted(set(dbscan_labels)):
    mask = dbscan_labels == label
    colour = 'crimson' if label == -1 else cluster_cmap(label / max(n_dbscan_clusters, 1))
    marker = 'x' if label == -1 else 'o'
    lbl = 'noise' if label == -1 else f'cluster {label}'
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[colour], s=60, marker=marker, label=lbl, zorder=5)
for i, name in enumerate(feat_df['cmake_target']):
    ax.annotate(name, (X_pca2[i, 0], X_pca2[i, 1]),
                textcoords='offset points', xytext=(3, 3), fontsize=6, ha='left')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'DBSCAN (eps={eps_suggested:.2f})')
ax.legend(fontsize=7)

# Hierarchical
ax = axes[1]
for label in sorted(set(hclust_labels)):
    mask = hclust_labels == label
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cmap(label / 9)], s=60, label=f'cluster {label}', zorder=5)
for i, name in enumerate(feat_df['cmake_target']):
    ax.annotate(name, (X_pca2[i, 0], X_pca2[i, 1]),
                textcoords='offset points', xytext=(3, 3), fontsize=6, ha='left')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'Hierarchical Ward (n={n_hclust})')
ax.legend(fontsize=7)

# Communities
ax = axes[2]
ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=comm_colours, s=60, zorder=5)
for i, name in enumerate(feat_df['cmake_target']):
    ax.annotate(name, (X_pca2[i, 0], X_pca2[i, 1]),
                textcoords='offset points', xytext=(3, 3), fontsize=6, ha='left')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'{comm_algo} Communities (n={n_communities})')
ax.legend(handles=comm_legend, fontsize=7, loc='best')

plt.tight_layout()
plt.show()

# ── Dendrogram ──────────────────────────────────────────────────────────────
Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
# Colour threshold: merge distance at n_communities cut
colour_threshold = Z[-(n_hclust - 1), 2] if len(Z) >= n_hclust else None
dendrogram(Z, labels=feat_df['cmake_target'].tolist(), leaf_rotation=45, leaf_font_size=9,
           color_threshold=colour_threshold, ax=ax)
ax.set_ylabel('Ward Distance')
ax.set_title(f'Hierarchical Clustering Dendrogram (colour cut at {n_hclust} clusters)')
plt.tight_layout()
plt.show()

# ── Cluster assignment table ────────────────────────────────────────────────
display(feat_df[['cmake_target', 'target_type', 'community', 'dbscan_label', 'hcluster']]
        .sort_values('hcluster').reset_index(drop=True))

## Interpretation

Centroid feature values per cluster. Codegen-heavy, template-heavy, include-bloated clusters.


In [ ]:
# Using hierarchical clustering for interpretation (DBSCAN assigns many points
# as noise, making centroid analysis less meaningful).

# ── Feature-to-label mapping for auto-naming clusters ───────────────────────
FEATURE_LABELS = {
    'compile_time_sum_ms': 'high-compile-cost',
    'code_lines_total': 'large-codebase',
    'file_count': 'many-files',
    'header_depth_max': 'deep-headers',
    'preprocessed_bytes_total': 'large-preprocessed',
    'object_size_total_bytes': 'large-objects',
    'direct_dependency_count': 'high-deps',
    'transitive_dependency_count': 'high-transitive-deps',
    'direct_dependant_count': 'high-fan-in',
    'git_commit_count_total': 'high-churn',
    'link_time_ms': 'slow-link',
    'codegen_ratio': 'codegen-heavy',
    'codegen_compile_time_sum_ms': 'codegen-compile-heavy',
    'expansion_ratio_mean': 'include-bloated',
    'gcc_template_time_sum_ms': 'template-heavy',
}

# ── Z-score centroid matrix ─────────────────────────────────────────────────
scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)
scaled_df['hcluster'] = hclust_labels

centroid_z = scaled_df.groupby('hcluster')[FEATURES].mean()

# ── Auto-name each cluster by its top dominant feature ──────────────────────
cluster_names = {}
for cid in centroid_z.index:
    row = centroid_z.loc[cid]
    # Skip zero-variance features for naming
    non_zero_var = [f for f in FEATURES if f not in zero_var]
    if non_zero_var:
        top_feat = max(non_zero_var, key=lambda f: abs(row[f]))
        top_z = row[top_feat]
        sign = '+' if top_z > 0 else '−'
        label = FEATURE_LABELS.get(top_feat, top_feat)
        cluster_names[cid] = f"C{cid}: {label} ({sign}{abs(top_z):.1f}σ)"
    else:
        cluster_names[cid] = f"C{cid}: baseline"

feat_df['hcluster_name'] = feat_df['hcluster'].map(cluster_names)

# ── Heatmap of z-score centroids ────────────────────────────────────────────
short_names = [f.replace('_ms', '').replace('_total', '').replace('_sum', '')
               .replace('_count', '').replace('_bytes', '') for f in FEATURES]

fig, ax = plt.subplots(figsize=(16, max(4, n_hclust * 1.2)))
heatmap_data = centroid_z.copy()
heatmap_data.index = [cluster_names[cid] for cid in heatmap_data.index]
heatmap_data.columns = short_names

sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            vmin=-2, vmax=2, linewidths=0.5, ax=ax)
ax.set_title('Cluster Feature Profiles (z-score, hierarchical Ward)')
ax.set_xlabel('Feature')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

# ── Cluster size + target type composition ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Cluster sizes
size_df = feat_df.groupby('hcluster_name').size().reset_index(name='count')
size_df = size_df.sort_values('count', ascending=True)
ax1.barh(size_df['hcluster_name'], size_df['count'], color='steelblue')
ax1.set_xlabel('Target Count')
ax1.set_title('Cluster Sizes')

# Target type composition
ct = pd.crosstab(feat_df['hcluster_name'], feat_df['target_type'])
ct.plot(kind='barh', stacked=True, ax=ax2, colormap='tab10')
ax2.set_xlabel('Target Count')
ax2.set_title('Target Type Composition per Cluster')
ax2.legend(title='Target Type', fontsize=7, loc='best')

plt.tight_layout()
plt.show()

# ── Raw centroid means table ────────────────────────────────────────────────
raw_df = pd.DataFrame(X_raw, columns=FEATURES)
raw_df['hcluster'] = hclust_labels
raw_centroids = raw_df.groupby('hcluster')[FEATURES].mean()
raw_centroids.index = [cluster_names[cid] for cid in raw_centroids.index]
print("Raw feature means per cluster:")
display(raw_centroids.T.round(1))

# ── Per-cluster summaries ───────────────────────────────────────────────────
print("\nCluster summaries:")
for cid in sorted(centroid_z.index):
    members = feat_df[feat_df['hcluster'] == cid]['cmake_target'].tolist()
    row = centroid_z.loc[cid]
    non_zero = [f for f in FEATURES if f not in zero_var]
    top2 = sorted(non_zero, key=lambda f: abs(row[f]), reverse=True)[:2]

    print(f"\n{cluster_names[cid]}")
    print(f"  Members ({len(members)}): {', '.join(sorted(members))}")
    for f in top2:
        raw_val = raw_centroids.loc[cluster_names[cid], f]
        print(f"  {f}: z={row[f]:+.2f} (raw mean={raw_val:.1f})")

# Flag zero-variance features
if zero_var:
    print(f"\nNote: {', '.join(zero_var)} has zero variance across all targets.")
    print("No template-heavy cluster can be identified from this build's data.")

## Outlier Detection

DBSCAN noise points — individual attention needed.


In [ ]:
# ── Identify DBSCAN noise points as outliers ────────────────────────────────
noise_mask = dbscan_labels == -1
cluster_mask = ~noise_mask

if noise_mask.sum() == 0:
    print("No DBSCAN noise points — all targets assigned to clusters.")
    print("Consider lowering eps to increase sensitivity.")
else:
    # ── Isolation score: distance to nearest clustered point ────────────────
    if cluster_mask.sum() > 0:
        nn_cluster = NearestNeighbors(n_neighbors=1).fit(X_scaled[cluster_mask])
        dists, _ = nn_cluster.kneighbors(X_scaled[noise_mask])
        isolation_scores = dists.ravel()
    else:
        isolation_scores = np.full(noise_mask.sum(), np.inf)

    # ── Auto-generate attention reasons from extreme features ───────────────
    noise_indices = np.where(noise_mask)[0]
    reasons = []
    for idx, iso_score in zip(noise_indices, isolation_scores):
        z_scores = X_scaled[idx]
        non_zero = [i for i, f in enumerate(FEATURES) if f not in zero_var]
        if non_zero:
            top2_fi = sorted(non_zero, key=lambda i: abs(z_scores[i]), reverse=True)[:2]
            parts = []
            for fi in top2_fi:
                raw_val = X_raw[idx, fi]
                sign = 'high' if z_scores[fi] > 0 else 'low'
                parts.append(f"{sign} {FEATURES[fi]} ({raw_val:.0f})")
            reasons.append('; '.join(parts))
        else:
            reasons.append('uniform profile')

    # ── Build outlier table ─────────────────────────────────────────────────
    outlier_df = feat_df.loc[noise_indices, ['cmake_target', 'target_type']].copy()
    outlier_df['isolation_score'] = isolation_scores
    outlier_df['compile_time_sum_ms'] = feat_df.loc[noise_indices, 'compile_time_sum_ms'].values
    outlier_df['codegen_ratio'] = feat_df.loc[noise_indices, 'codegen_ratio'].values
    outlier_df['header_depth_max'] = feat_df.loc[noise_indices, 'header_depth_max'].values
    outlier_df['expansion_ratio_mean'] = feat_df.loc[noise_indices, 'expansion_ratio_mean'].values
    outlier_df['attention_reason'] = reasons
    outlier_df = outlier_df.sort_values('isolation_score', ascending=False).reset_index(drop=True)

    # ── Plots ───────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # PCA scatter: outliers vs clustered
    ax1.scatter(X_pca2[cluster_mask, 0], X_pca2[cluster_mask, 1],
                c='steelblue', s=60, label='Clustered', zorder=4)
    ax1.scatter(X_pca2[noise_mask, 0], X_pca2[noise_mask, 1],
                c='crimson', s=120, marker='X', label='Outlier (noise)', zorder=5)
    for i, name in enumerate(feat_df['cmake_target']):
        ax1.annotate(name, (X_pca2[i, 0], X_pca2[i, 1]),
                     textcoords='offset points', xytext=(3, 3), fontsize=7, ha='left')
    ax1.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
    ax1.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
    ax1.set_title(f'DBSCAN Outliers (eps={eps_suggested:.2f}, {noise_mask.sum()} noise points)')
    ax1.legend()

    # k-distance curve with eps threshold
    ax2.plot(range(1, len(k_distances) + 1), k_distances, 'o-', color='steelblue', markersize=4)
    ax2.axhline(eps_suggested, color='red', linestyle='--', alpha=0.6,
                label=f'eps = {eps_suggested:.2f} ({noise_mask.sum()} noise)')
    ax2.set_xlabel('Points (sorted by distance)')
    ax2.set_ylabel(f'{k}-th Nearest Neighbour Distance')
    ax2.set_title('k-Distance Curve with eps Threshold')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    # ── Display outlier table ───────────────────────────────────────────────
    print(f"DBSCAN noise points: {len(outlier_df)} targets require individual attention\n")
    display(outlier_df)

    # ── Summary ─────────────────────────────────────────────────────────────
    print("\nOutlier details (ranked by isolation score):")
    for _, row in outlier_df.iterrows():
        print(f"  {row['cmake_target']:30s} [isolation={row['isolation_score']:.2f}]: "
              f"{row['attention_reason']}")

    print("\nRecommendation: inspect each outlier target individually before applying")
    print("cluster-level optimisation strategies — their unique feature combinations")
    print("mean cluster-derived recommendations may not apply.")